# FrozenSE — experiments notebook (v1.6.2)

Trains and evaluates every model of the FrozenSE study except nnU-Net (see
`FrozenSE_SAM_nnUNet.ipynb`) by leave-one-slide-out cross-validation on the
193 labelled Gleason 2019 cores. It runs on **Kaggle** (GPU T4 × 2) or
**Colab**. Every fold is saved when it finishes, so a long stage can be split
across sessions: run the same stage again with the previous output attached
and it continues where it stopped.

Step-by-step instructions are in `docs/REPRODUCE.md` of the repository.


## 0 · Choose what this run does

Set `STAGE`, then run the whole notebook (*Run all* on Colab; *Save Version →
Save & Run All* on Kaggle, which keeps running with the browser closed).
Everything outside the chosen stage is skipped, so *Run all* never starts
hours of training you did not ask for.

| `STAGE` | Does | GPU |
|---|---|---|
| `'setup'` | data checks, label audit, consensus and inter-annotator agreement, split preview | **not needed** |
| `'smoke'` | a short fold-0 run and a GPU-hours estimate for every stage | needed |
| `'baselines'` | the main comparison: 4 models × 6 folds | needed |
| `'ablations'` | component and loss ablations, 3 folds each | needed |
| `'improve'` | the improved model (2 variants, chosen on validation) and both U-Nets retrained under the same protocol: 4 models × 6 folds | needed |
| `'data_efficiency'` | the learning curve | needed |
| `'extra'` | 5-core runs scored on test slides (3 draws) and the same decoder on pathology encoders (Phikon; UNI if `HF_TOKEN` is set) | needed |
| `'report'` | every table from saved results; trains nothing | not needed |

A training stage that does not finish in one session just stops before the
time limit with every completed fold saved. Run the same stage again and it
continues.


In [ ]:
# ===================== CHOOSE WHAT THIS RUN DOES =====================
STAGE = 'setup'
# =====================================================================

import os, sys, time, glob, json, shutil, zipfile, importlib, importlib.util
SESSION_START = time.time()
TRAINING_STAGES = {'smoke', 'baselines', 'improve', 'ablations', 'data_efficiency', 'extra'}
assert STAGE in TRAINING_STAGES | {'setup', 'report'}, f"unknown STAGE {STAGE!r}"

ON_KAGGLE = os.path.isdir('/kaggle/input')
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
# Kaggle's newer images ship the google.colab package too, so importing it
# proves nothing there: Kaggle wins.
ON_COLAB = ON_COLAB and not ON_KAGGLE

if ON_KAGGLE:
    PLATFORM    = 'kaggle'
    SEARCH_ROOT = '/kaggle/input'                  # your datasets + earlier outputs
    SCRATCH     = '/kaggle/tmp'                    # working space, not saved
    OUTPUT_ROOT = '/kaggle/working/frozense_runs'  # saved with each version
    # A Kaggle session ends at 12 hours, and output from a session killed at
    # the limit may never be saved. So no fold is STARTED unless it is
    # expected to finish within 11.5 hours; the run then ends cleanly and
    # the next session continues.
    DEADLINE = SESSION_START + 11.5 * 3600
elif ON_COLAB:
    PLATFORM    = 'colab'
    SEARCH_ROOT = '/content/drive/MyDrive/frozense'
    SCRATCH     = '/content'
    OUTPUT_ROOT = '/content/drive/MyDrive/frozense_runs'
    DEADLINE    = None   # each fold is written to Drive the moment it ends
else:
    PLATFORM    = 'local'
    SEARCH_ROOT = os.path.abspath('frozense_inputs')
    SCRATCH     = os.path.abspath('frozense_scratch')
    OUTPUT_ROOT = os.path.abspath('frozense_runs')
    DEADLINE    = None

os.makedirs(SCRATCH, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"platform: {PLATFORM}   STAGE: {STAGE!r}")
print(f"inputs  : {SEARCH_ROOT}")
print(f"outputs : {OUTPUT_ROOT}")

## 1 · Environment

Detects Kaggle or Colab, sets the input and output folders and records the session start time.


In [ ]:
!pip install -q "segmentation_models_pytorch==0.6.0" "albumentations==2.0.8" "timm>=1.0" git+https://github.com/ChaoningZhang/MobileSAM.git

In [ ]:
missing = [m for m in ('segmentation_models_pytorch', 'albumentations', 'timm', 'mobile_sam')
           if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(
        f"Not installed: {missing}. On Kaggle, turn on Settings -> Internet "
        "(it needs a verified phone number), then run the cell above again.")

import torch, numpy as np
import segmentation_models_pytorch as smp, albumentations as A, timm
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"smp {smp.__version__} | albumentations {A.__version__} | timm {timm.__version__}")

# Check the GPU with a REAL operation, not just by asking whether one exists.
# Kaggle's P100 currently passes the existence check but fails on its first
# computation with "no kernel image is available", because the PyTorch build
# Kaggle ships no longer supports that GPU generation.
GPU_OK, gpu_problem = False, None
if torch.cuda.is_available():
    try:
        x = torch.randn(2, 3, 64, 64, device='cuda', requires_grad=True)
        w = torch.randn(4, 3, 3, 3, device='cuda')
        torch.nn.functional.conv2d(x, w).sum().backward()
        torch.cuda.synchronize()
        GPU_OK = True
        print("GPU:", torch.cuda.get_device_name(0))
    except Exception as exc:
        gpu_problem = f"{type(exc).__name__}: {exc}"

DEVICE = torch.device('cuda' if GPU_OK else 'cpu')
NUM_WORKERS = min(4, os.cpu_count() or 2)

if STAGE in TRAINING_STAGES and not GPU_OK:
    raise RuntimeError(
        f"STAGE {STAGE!r} trains models and needs a working GPU. "
        + (f"The GPU failed a test operation ({gpu_problem}). " if gpu_problem else
           "No GPU is attached. ")
        + "On Kaggle: Settings -> Accelerator -> GPU T4 x2 (not P100). "
          "On Colab: Runtime -> Change runtime type -> GPU.")
if not GPU_OK:
    print(f"No GPU in use -- fine for STAGE {STAGE!r}.")

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

### Install the `frozense` package

Finds the `frozense` package among the attached inputs (a folder or a zip), copies it to scratch space and imports it. The printed version should be 1.6.2.


In [ ]:
PKG_ROOT = os.path.join(SCRATCH, 'frozense_pkg')


def safe_extract(zip_path, dest):
    # Extract a zip, accepting EITHER path separator.
    #
    # Before version 1.2.3 of its Archive module, Windows PowerShell's
    # Compress-Archive wrote entries with backslashes
    # ('imagesTr\slide001_core004.jpg'). On Linux, Python treats a backslash
    # as an ordinary character, so a plain extractall() leaves every file
    # sitting flat in one folder with backslashes in its NAME -- and every
    # later search for imagesTr/*.jpg finds nothing.
    dest_real = os.path.realpath(dest)
    n = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/').lstrip('/')
            if not name or name.endswith('/'):
                continue                      # directory entry
            target = os.path.realpath(os.path.join(dest, name))
            if not target.startswith(dest_real + os.sep):
                raise ValueError(f"refusing to extract outside {dest}: {info.filename}")
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as out:
                shutil.copyfileobj(src, out, 1 << 20)
            n += 1
    return n


def find_files(pattern):
    return sorted(glob.glob(f'{SEARCH_ROOT}/**/{pattern}', recursive=True),
                  key=os.path.getmtime, reverse=True)


if not os.path.isdir(SEARCH_ROOT):
    raise FileNotFoundError(
        f"{SEARCH_ROOT} does not exist. " +
        ("Create a folder named 'frozense' directly inside My Drive and put "
         "gleason2019.zip, mobile_sam.pt and FrozenSE_SAM_Revision_Package.zip in it."
         if ON_COLAB else "Add your datasets to the notebook (right panel -> Add Input)."))

pkg_zips = find_files('FrozenSE_SAM_Revision_Package*.zip')
pkg_dirs = sorted(os.path.dirname(p) for p in
                  glob.glob(f'{SEARCH_ROOT}/**/frozense/__init__.py', recursive=True))

shutil.rmtree(PKG_ROOT, ignore_errors=True)
if pkg_zips:
    PKG_SOURCE = pkg_zips[0]
    tmp = os.path.join(SCRATCH, '_pkg_unzip')
    shutil.rmtree(tmp, ignore_errors=True)
    safe_extract(PKG_SOURCE, tmp)
    inner = glob.glob(f'{tmp}/**/frozense/__init__.py', recursive=True)
    if not inner:
        raise FileNotFoundError(f"{PKG_SOURCE} contains no frozense/ package")
    shutil.copytree(os.path.dirname(inner[0]), f'{PKG_ROOT}/frozense')
    shutil.rmtree(tmp, ignore_errors=True)
elif pkg_dirs:
    PKG_SOURCE = pkg_dirs[0]
    shutil.copytree(PKG_SOURCE, f'{PKG_ROOT}/frozense')
else:
    raise FileNotFoundError(
        f"No frozense package anywhere under {SEARCH_ROOT}. Upload "
        "FrozenSE_SAM_Revision_Package.zip " +
        ("into that Drive folder." if ON_COLAB else "as a Kaggle dataset and add it as an input."))

# After a FAILED import, Python remembers 'nothing here' for this path, so
# copying the package in is not enough on its own; after an earlier
# SUCCESSFUL one, old modules stay loaded. Clear both.
for name in [m for m in sys.modules if m == 'frozense' or m.startswith('frozense.')]:
    del sys.modules[name]
sys.path_importer_cache.pop(PKG_ROOT, None)
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)
importlib.invalidate_caches()

import frozense
print(f"frozense {frozense.__version__}")
print(f"  source : {PKG_SOURCE}")
print(f"  loaded : {os.path.dirname(frozense.__file__)}")

In [ ]:
# Results from earlier sessions. On Colab they are already in Drive. On
# Kaggle they arrive as the attached output of your previous version (right
# panel -> Add Input -> Your Work -> this notebook), packaged as
# frozense_runs.zip; they are unpacked into this session's output so that
# finished folds are loaded instead of retrained.
if ON_KAGGLE:
    for z in sorted(find_files('*frozense[-_]runs*.zip'), key=os.path.getmtime):
        safe_extract(z, OUTPUT_ROOT)
    for d in glob.glob(f'{SEARCH_ROOT}/**/*frozense[-_]runs*', recursive=True):
        if os.path.isdir(d):
            shutil.copytree(d, OUTPUT_ROOT, dirs_exist_ok=True)

saved = glob.glob(f'{OUTPUT_ROOT}/**/fold_result.json', recursive=True)
print(f"{len(saved)} saved fold results available from earlier sessions")

## 2 · Data

Inputs the notebook expects to find among the attached datasets:

- the Gleason 2019 folders `imagesTr`, `imagesVal`, `Generated_Labels_Train`,
  `labelsVal` and `annotations` (one subfolder per pathologist, `Map1` to
  `Map6`), unpacked or as one zip;
- `mobile_sam.pt`, the official MobileSAM weights;
- the `frozense` package (folder or zip).

On Colab, put the same files in `MyDrive/frozense/`. The cell below handles
data that is unpacked, zipped, or unpacked flat with backslashes in the file
names (a Windows-made zip opened on Linux). Expected: 55 + 138 = **193
labelled cores**.


In [ ]:
FOLDERS = ('imagesTr', 'imagesVal', 'Generated_Labels_Train', 'labelsVal', 'annotations')
RAW = os.path.join(SCRATCH, 'raw')
MARK = os.path.join(RAW, '.prepared_from')

weights = find_files('mobile_sam*.pt')
if not weights:
    raise FileNotFoundError(f"No mobile_sam.pt anywhere under {SEARCH_ROOT}")
WEIGHTS = weights[0]


def locate(root):
    out = {}
    for name in FOLDERS:
        hits = sorted(d for d in glob.glob(f'{root}/**/{name}', recursive=True)
                      if os.path.isdir(d))
        if hits:
            out[name] = hits[0]
    return out


def flat_backslash_files(root):
    found = []
    for r, _, files in os.walk(root):
        found += [os.path.join(r, f) for f in files
                  if '\\' in f and f.split('\\')[0] in FOLDERS]
    return found


DATA, how = {}, None
if ON_KAGGLE:
    in_place = locate(SEARCH_ROOT)
    if len(in_place) == len(FOLDERS):
        DATA, how = in_place, 'read in place from your Kaggle dataset'

if not DATA:
    flat = flat_backslash_files(SEARCH_ROOT)
    zips = find_files('gleason2019*.zip')
    source = (f'flat:{os.path.dirname(flat[0])}' if flat else
              f'zip:{zips[0]}' if zips else None)
    if source is None:
        raise FileNotFoundError(
            f"Found neither the labelled folders, nor gleason2019.zip, under "
            f"{SEARCH_ROOT}. Check the inputs are attached.")

    done = os.path.exists(MARK) and open(MARK).read() == source
    if not done:
        # The marker is written only AFTER a complete job, so a session that
        # dropped mid-way is redone rather than mistaken for finished.
        shutil.rmtree(RAW, ignore_errors=True)
        os.makedirs(RAW)
        if flat:
            print(f"rebuilding folders for {len(flat)} files unpacked flat ...")
            for f in flat:
                rel = os.path.basename(f).split('\\')
                dest = os.path.join(RAW, *rel)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy(f, dest)
        else:
            print("copying the dataset zip to local disk (a few minutes) ...")
            local = os.path.join(SCRATCH, 'gleason.zip')
            shutil.copy(zips[0], local)
            print("extracting ...")
            safe_extract(local, RAW)
            os.remove(local)
        with open(MARK, 'w') as fh:
            fh.write(source)
    DATA, how = locate(RAW), ('rebuilt from flat files' if flat else 'extracted from the zip')

absent = [f for f in FOLDERS if f not in DATA]
if absent:
    raise FileNotFoundError(f"Missing folders: {absent}. Found: {DATA}")

print(f"data    : {how}")
print(f"weights : {WEIGHTS}\n")
for name in FOLDERS:
    if name == 'annotations':
        maps = sorted(d for d in os.listdir(DATA[name]) if os.path.isdir(os.path.join(DATA[name], d)))
        print(f"  {name:<24}{len(maps):>5} reader folders ({', '.join(maps)})")
    else:
        print(f"  {name:<24}{len(os.listdir(DATA[name])):>5} files")

### Index every core where it lies

Builds one index of all labelled cores, with the slide each belongs to. Expected: 193 cores on 6 slides.


In [ ]:
IMAGE_DIRS = [DATA['imagesTr'], DATA['imagesVal']]
LABEL_DIRS = [DATA['Generated_Labels_Train'], DATA['labelsVal']]
ANN_DIR = DATA['annotations']

SOURCE_OF, LBL_SOURCE, clashes = {}, {}, []


def register(folder_name, registry, exts, strip=''):
    for f in sorted(os.listdir(DATA[folder_name])):
        stem, ext = os.path.splitext(f)
        if ext.lower() not in exts:
            continue
        core_id = stem[:-len(strip)] if strip and stem.endswith(strip) else stem
        if core_id in registry:
            clashes.append((core_id, registry[core_id], folder_name))
        registry[core_id] = folder_name


for name in ('imagesTr', 'imagesVal'):
    register(name, SOURCE_OF, {'.jpg', '.jpeg', '.png'})
for name in ('Generated_Labels_Train', 'labelsVal'):
    register(name, LBL_SOURCE, {'.png'}, strip='_classimg_nonconvex')

print(f"images: {len(SOURCE_OF)} cores   labels: {len(LBL_SOURCE)} cores")
if clashes:
    # The same core in two source folders means they overlap, which would be
    # worth knowing before trusting any of the original numbers.
    print(f"WARNING: {len(clashes)} core(s) appear in two source folders, "
          f"e.g. {clashes[:3]}")

### 2b · Audit the labels before trusting anything

Checks label values and foreground fractions on a sample of cores and reports cores with no tumour pixels.


In [ ]:
if PKG_ROOT not in sys.path:   # installed in step 1
    sys.path.insert(0, PKG_ROOT)
from frozense.config import *
from frozense.data import (
    discover_cores, make_split, n_folds, PatchDataset,
    audit_label_values, format_label_audit, group_composition,
)

base = ExperimentConfig(
    name='proposed_frozense_sam',
    split=SplitConfig(
        group_regex=r'^(?P<group>slide\d+)',   # matches slide001_core004.jpg
        allow_core_level_fallback=False,
        scheme='leave_one_group_out',
        n_train_cores=35, seed=42,
    ),
    data=DataConfig(
        images_dir=IMAGE_DIRS, labels_dir=LABEL_DIRS,   # read in place, both folders
        # slide001_core004.jpg  ->  slide001_core004_classimg_nonconvex.png
        label_suffixes=('_classimg_nonconvex', ''),
        tumour_grades=(3, 4, 5),               # <-- the audit below checks this
    ),
    model=ModelConfig(sam_checkpoint=WEIGHTS),
    output_root=OUTPUT_ROOT,
)
# Training budget. 30 epochs with patience 10 fits the free Kaggle quota;
# every model gets the same budget, so comparisons stay fair. State it in
# the methods. (The library default is 300 / 40.)
base = replace_nested(base, **{'train.num_workers': NUM_WORKERS,
                               'train.epochs': 30, 'train.early_stopping_patience': 10})

# The final model, chosen on validation in step 5b. Ablations and the
# learning curve are run on THIS design, not the original.
from frozense.experiments import IMPROVED_PROTOCOL
from frozense.experiments import build_improvements as _bi
UNET_R2 = next(e.cfg for e in _bi(base) if e.cfg.name == 'baseline_unet_pretrained_r2')
FINAL = replace_nested(base, name='improved_skip_partial',
                       **{'model.use_skip_connections': True,
                          'model.unfreeze_last_n_blocks': 1}, **IMPROVED_PROTOCOL)

cores = discover_cores(base.data, base.split)
print(f"{len(cores)} cores, {n_folds(cores)} slides\n")

comp = group_composition(cores, SOURCE_OF)
for slide, info in sorted(comp['per_group'].items()):
    src = ', '.join(f"{k} {v}" for k, v in sorted(info['sources'].items()))
    print(f"  {slide}: {info['n_cores']:>3} cores   ({src})")

shared = comp['groups_spanning_multiple_sources']
print(f"\nslides present in BOTH original source folders: {shared or 'none'}")
if shared:
    print("  -> the released imagesTr/imagesVal partition is NOT slide-disjoint.")
    print("     Same staining batch in training and in the set used for both")
    print("     checkpoint selection and final reporting.")

### The extra experiments (`STAGE = 'extra'`)

Defines (i) the 5-core experiment: FrozenSE-SAM and the ImageNet U-Net trained
on five cores with 245 patches each (1,225 patches per epoch, as with 35
cores), three independent draws of the cores, all six folds, scored on test
slides; and (ii) the pathology encoders: the same decoder and protocol on
Phikon (public) and UNI (gated: add your Hugging Face token as a secret
labelled `HF_TOKEN`; without it UNI is skipped and everything else runs).


In [ ]:
LOWDATA_DRAWS = (42, 1042, 2042)

def lowdata(cfg, draw, tag):
    return replace_nested(cfg, name=f'lowdata5_{tag}_d{draw}',
                          **{'split.n_train_cores': 5, 'split.seed': draw,
                             'data.patches_per_core_train': 245})

PFM_PHIKON = replace_nested(FINAL, name='pfm_phikon',
                            **{'model.arch': 'pfm_phikon', 'model.unfreeze_last_n_blocks': 2})
PFM_UNI = replace_nested(FINAL, name='pfm_uni',
                         **{'model.arch': 'pfm_uni', 'model.unfreeze_last_n_blocks': 2})

UNI_OK = os.path.isdir(f'{OUTPUT_ROOT}/pfm_uni')       # earlier sessions already ran it
if STAGE == 'extra':
    token = os.environ.get('HF_TOKEN')
    if not token and ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if token:
        os.environ['HF_TOKEN'] = token
    # Download here, once, so the two GPU workers do not race for the same files.
    from huggingface_hub import snapshot_download
    snapshot_download('owkin/phikon')
    if token:
        try:
            snapshot_download('MahmoodLab/uni', token=token)
            UNI_OK = True
        except Exception as e:
            print(f"UNI not available ({type(e).__name__}); Phikon and the 5-core runs go ahead.")
    else:
        print("No HF_TOKEN secret: UNI skipped; Phikon and the 5-core runs go ahead.")

# Priority order, so a session cut short leaves the most important runs done:
# draw 1 of both models, Phikon, draw 2, UNI, then draw 3.
def draw(d):
    return [(lowdata(FINAL, d, 'frozense'), 1.7, False), (lowdata(UNET_R2, d, 'unet'), 0.7, False)]
EXTRA = draw(LOWDATA_DRAWS[0]) + [(PFM_PHIKON, 1.4, True)] + draw(LOWDATA_DRAWS[1])
if UNI_OK:
    EXTRA.append((PFM_UNI, 2.6, True))
EXTRA += draw(LOWDATA_DRAWS[2])
print("extra experiments:", ", ".join(c.name for c, _, _ in EXTRA))

In [ ]:
# Raises if any sampled core binarises to an empty mask under tumour_grades.
report = audit_label_values(cores, base.data, source_of=LBL_SOURCE)
print(format_label_audit(report))

### 2c · The per-pathologist annotation maps

Reads the per-pathologist maps (`Map1` to `Map6`), counts readers per core, and tests which vote order and tie rule reproduce the consensus labels in use.


In [ ]:
from frozense.consensus import (
    discover_annotations, coverage_report, format_coverage,
    agreement_report, format_agreement,
    build_consensus, compare_to_existing, format_reproducibility,
)

print("annotations tree:", ANN_DIR)
ann = discover_annotations(ANN_DIR, label_suffixes=base.data.label_suffixes)
cov = coverage_report(ann, [c.core_id for c in cores])
print()
print(format_coverage(cov))

In [ ]:
# Can the labels being trained on be regenerated from the maps?
# (Runs in the 'setup' stage only: it reads hundreds of full-size maps, which
# is CPU work that would otherwise burn GPU quota in every training session.)
if STAGE == 'setup':
    existing = {c.core_id: c.label_path for c in cores}
    repro = compare_to_existing(ann, base.data, existing, max_cores=15)
    print(format_reproducibility(repro))
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'setup'")

In [ ]:
# Inter-annotator agreement. downsample=4 for speed; set 1 for the exact
# figure, which differs only in the third decimal. 'setup' stage only.
if STAGE == 'setup':
    agr = agreement_report(ann, base.data, max_cores=40, downsample=4)
    print(format_agreement(agr))
    with open(f'{OUTPUT_ROOT}/inter_annotator_agreement.json', 'w') as fh:
        json.dump(agr.summary(), fh, indent=2, default=str)
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'setup'")

### Does reader coverage track the slide?

Reader counts per slide, to check that no fold is evaluated on systematically thinner annotation.


In [ ]:
from frozense.consensus import readers_by_group, format_readers_by_group

rbg = readers_by_group(ann, core_ids=[c.core_id for c in cores],
                       group_regex=base.split.group_regex)
print(format_readers_by_group(rbg))

### Optional: a uniform reader subset

Finds the largest set of pathologists who read nearly every core, which gives one fixed majority rule. Reported as a sensitivity analysis only.


In [ ]:
from frozense.consensus import uniform_reader_subset, format_uniform_subset

subsets = uniform_reader_subset(ann, core_ids=[c.core_id for c in cores])
print(format_uniform_subset(subsets, total=len(cores)))

# To use one, pass its readers to build_consensus:
#   build_consensus(ann, core_id, base.data, annotators=subsets[1]['readers'])

### Per-core consensus table

For every core: which pathologists annotated it and which majority rule applied.


In [ ]:
from frozense.consensus import consensus_manifest, summarise_manifest
import pandas as pd

# Builds a consensus mask for every core at full resolution: 'setup' only.
if STAGE == 'setup':
    man = consensus_manifest(ann, base.data,
                             core_ids=[c.core_id for c in cores],
                             vote_order='binarise_then_vote', tie_policy='benign')
    print(summarise_manifest(man))
    man_df = pd.DataFrame(man)
    man_df.to_csv(f'{OUTPUT_ROOT}/table_consensus_manifest.csv', index=False)
    display(man_df.head(10))
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'setup'")

## 3 · The split: leave-one-slide-out

Fold *k* tests on slide *k*, validates on the next slide and trains on 35 cores drawn from the other four. The validation slide drives early stopping, scheduling and checkpoint choice; `TestSetGuard` lets each test slide be read once. Every core is tested exactly once (pooled n = 193). Grouping is by slide; the release has no core-to-patient key.


In [ ]:
import json

for fold in range(n_folds(cores)):
    sp = make_split(cores, replace_nested(base, **{'split.fold': fold}).split)
    m = sp.manifest()
    print(f"fold {fold}:  test {m['groups']['test'][0]} ({m['n_cores']['test']:>2} cores)"
          f"   val {m['groups']['val'][0]} ({m['n_cores']['val']:>2})"
          f"   train {m['n_cores']['train']} of {m['train_pool_size']} available"
          f" from {m['n_groups']['train']} slides")

split = make_split(cores, base.split)          # fold 0, for the smoke test below
print(json.dumps({k: v for k, v in split.manifest().items() if k != 'cores'}, indent=2))

## 4 · Smoke test — fold 0, short run, validation only

A short fold-0 run that checks the whole pipeline and prints a measured GPU-hours estimate for every stage. It never reads a test slide.


In [ ]:
from frozense.models import build_model, count_parameters
from frozense.engine import fit
from frozense.losses import describe_loss

if STAGE == 'smoke':
    print("Loss actually optimised:\n ", describe_loss(base.loss))
    smoke = replace_nested(base, name='smoke_test',
                           **{'train.epochs': 6, 'train.early_stopping_patience': 6})

    t0 = time.time()
    train_ds = PatchDataset(split.train, smoke.data, smoke.aug, train=True,
                            seed=smoke.train.seed, verbose=True)
    val_ds = PatchDataset(split.val, smoke.data, smoke.aug, train=False,
                          seed=smoke.train.seed, verbose=True)
    preload_s = time.time() - t0

    m = build_model(smoke.model, target_size=smoke.data.patch_size)
    print("parameters:", count_parameters(m))
    m, logs, h = fit(m, train_ds, val_ds, smoke, DEVICE)
    print(f"\nsmoke-test val Dice {h['best_val_score']:.4f} @ epoch {h['best_epoch']}")
    print("Pipeline runs. Nothing above touched a test set.")

    # ---- what each stage will cost, from the measured epoch time ----
    # Epoch 0 includes one-off warm-up, so it is left out of the average.
    per_epoch = sum(l.seconds for l in logs[1:]) / max(len(logs) - 1, 1)
    EPOCHS = min(100, base.train.epochs)   # assumption: early stopping (patience 40) usually ends runs
                   # well before the 300-epoch cap. Real runs will vary.
    run_h = (per_epoch * EPOCHS + 2 * preload_s) / 3600
    estimate = {
        # 6 proposed runs + 12 U-Net runs at roughly 0.7x the cost each
        'baselines':       run_h * (6 + 12 * 0.7),
        # 9 ablations x 3 folds; the two encoder-training ones cost ~2x;
        # the reference folds are reused from 'baselines'
        'ablations':       run_h * (9 * 3 + 2 * 3),
        # sizes 5/10/20 x 3 folds at reduced cost; the 35-core point is
        # reused from 'baselines'
        'data_efficiency': run_h * 3.9,
    }
    n_cards = max(1, torch.cuda.device_count())
    print(f"\n{per_epoch:.0f} s per epoch -> about {run_h:.1f} h per training run "
          f"at {EPOCHS} epochs\n")
    print(f"  {'stage':<16}{'GPU-hours':>10}{'session hours on ' + str(n_cards) + ' card(s)':>30}")
    for stage, hours in estimate.items():
        print(f"  {stage:<16}{hours:>10.0f}{hours / n_cards:>30.0f}")
    total = sum(estimate.values())
    print(f"  {'total':<16}{total:>10.0f}{total / n_cards:>30.0f}")
    if ON_KAGGLE:
        print("\nKaggle's weekly GPU quota is shown in your account settings "
              "(commonly about 30 h). Check how much this smoke run used, to see "
              "whether Kaggle counts session hours or hours per card. Every "
              "stage resumes across sessions.")
    del m, train_ds, val_ds
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'smoke'")

## Using every GPU

With two GPUs the notebook starts one worker per card (`python -m frozense.runner`), each training one fold at a time. Every fold is saved the moment it finishes, and a stage re-run skips saved folds.


In [ ]:
import subprocess
from dataclasses import asdict
from frozense.runner import plan

ABLATION_FOLDS = (0, 2, 4)
ABLATION_PRIORITY = ['ablation_se_off', 'ablation_aux_sdf', 'loss_focal_tversky',
                     'loss_dice', 'loss_tversky_only', 'loss_bce', 'ablation_skip_on',
                     'ablation_encoder_partial', 'ablation_encoder_unfrozen']
CURVE_SIZES = (5, 10, 20, 35)
CURVE_FOLDS = (0, 2, 4)

N_GPUS = torch.cuda.device_count() if GPU_OK else 0
RAN_IN_WORKERS, WORKER_PROBLEMS = False, []


def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            for line in fh:
                if line.startswith('MemAvailable:'):
                    return int(line.split()[1]) / 1024 ** 2
    except OSError:
        pass
    return float('inf')


# Each worker holds its preloaded patches (~2 GB at full size), PyTorch and
# the CUDA libraries, and its data-loader processes: about 8 GB at peak, by
# measurement plus the size of the real data. Only as many workers as fit
# are started -- running out of RAM gets a worker killed mid-fold.
GB_PER_WORKER = 8
N_WORKERS = min(N_GPUS, int(available_gb() // GB_PER_WORKER))

if STAGE in ('baselines', 'improve', 'ablations', 'data_efficiency', 'extra') and N_GPUS >= 2 and N_WORKERS < 2:
    print(f"{N_GPUS} GPUs, but only {available_gb():.0f} GB of RAM free -- not enough "
          f"for {N_GPUS} workers at ~{GB_PER_WORKER} GB each, so this stage trains on "
          "one GPU in the notebook instead.")

if STAGE in ('baselines', 'improve', 'ablations', 'data_efficiency', 'extra') and N_WORKERS >= 2:
    stage_base = FINAL if STAGE in ('ablations', 'data_efficiency') else base
    per_worker = replace_nested(stage_base, **{'train.num_workers': max(1, NUM_WORKERS // N_WORKERS)})
    job = {'stage': STAGE, 'base': asdict(per_worker),
           'cores': [asdict(c) for c in cores], 'deadline': DEADLINE,
           'ablation_folds': ABLATION_FOLDS, 'ablation_priority': ABLATION_PRIORITY,
           'curve_sizes': CURVE_SIZES, 'curve_folds': CURVE_FOLDS,
           'curve_models': [asdict(replace_nested(m, **{'train.num_workers': max(1, NUM_WORKERS // N_WORKERS)}))
                            for m in (FINAL, UNET_R2)],
           'extra_models': [{'cfg': asdict(replace_nested(c, **{'train.num_workers': max(1, NUM_WORKERS // N_WORKERS)})),
                             'cost': cost, 'measure_eff': eff} for c, cost, eff in EXTRA]}
    job['shares'] = plan(job, per_worker, cores, N_WORKERS)
    print(f"{N_WORKERS} workers: " + ", ".join(f"card {i} gets {len(s)} unit(s)"
                                             for i, s in enumerate(job['shares'])), flush=True)

if STAGE in ('baselines', 'improve', 'ablations', 'data_efficiency', 'extra') and N_WORKERS >= 2 \
        and not any(job['shares']):
    RAN_IN_WORKERS = True
    print("Every unit of this stage is already saved -- nothing to train.")
elif STAGE in ('baselines', 'improve', 'ablations', 'data_efficiency', 'extra') and N_WORKERS >= 2:
    job_path = os.path.join(SCRATCH, 'job.json')
    with open(job_path, 'w') as fh:
        json.dump(job, fh, default=str)

    logs = [os.path.join(SCRATCH, f'worker{i}.log') for i in range(N_WORKERS)]
    procs = []
    for i in range(N_WORKERS):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i),
                   PYTHONPATH=os.pathsep.join(p for p in (PKG_ROOT, os.environ.get('PYTHONPATH')) if p))
        procs.append(subprocess.Popen(
            [sys.executable, '-u', '-m', 'frozense.runner', '--job', job_path,
             '--worker', str(i), '--n-workers', str(N_WORKERS)],
            stdout=open(logs[i], 'w'), stderr=subprocess.STDOUT, env=env, cwd=SCRATCH))

    # Stream both logs into this cell: progress to watch, and regular output
    # for Kaggle's unattended runner, which can treat a silent cell as dead.
    offsets, partial = [0] * N_WORKERS, [''] * N_WORKERS
    def pump():
        for i, path in enumerate(logs):
            with open(path) as fh:
                fh.seek(offsets[i])
                chunk = fh.read()
                offsets[i] = fh.tell()
            lines = (partial[i] + chunk).split('\n')
            partial[i] = lines.pop()
            for line in lines:
                print(f"[gpu {i}] {line}", flush=True)

    while any(p.poll() is None for p in procs):
        time.sleep(15)
        pump()
    pump()
    for i, rest in enumerate(partial):
        if rest:
            print(f"[gpu {i}] {rest}", flush=True)
    for i, p in enumerate(procs):
        if p.returncode != 0:
            WORKER_PROBLEMS.append(f"worker {i} exited with code {p.returncode}; "
                                   f"its log is {logs[i]}")
    RAN_IN_WORKERS = True
    print("\nworkers finished" + (": " + "; ".join(WORKER_PROBLEMS) if WORKER_PROBLEMS else
                                  " -- the cells below assemble their results"))
else:
    print(f"{N_GPUS} GPU(s) available; nothing to parallelise in STAGE {STAGE!r}.")

## 5 · Baselines, cross-validated

The first comparison: the fully frozen design, the ImageNet and scratch U-Nets and zero-shot MobileSAM, six folds each.


In [ ]:
from frozense.crossval import (
    run_cross_validation, run_learning_curve, config_fingerprint,
    cv_summary_table, cv_comparison_table, format_cv_table, format_comparison,
)
from frozense.experiments import build_baselines
import pandas as pd

cvs = {}
if STAGE in ('baselines', 'report'):
    for exp in build_baselines(base):
        print(f"\n{'=' * 70}\n{exp.cfg.name}\n{exp.purpose}\n{'=' * 70}", flush=True)
        cv = run_cross_validation(
            exp.cfg, cores, device=DEVICE, verbose=True,
            train_missing=(STAGE == 'baselines' and not RAN_IN_WORKERS), deadline=DEADLINE,
        )
        if cv is not None:
            cvs[exp.cfg.name] = cv
    print(f"\ncomplete: {sorted(cvs)}")
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'baselines' (trains) or 'report'")

In [ ]:
if cvs:
    rows = cv_summary_table(list(cvs.values()), base)
    print(format_cv_table(sorted(rows, key=lambda r: -r['Dice'])))
    df = pd.DataFrame(rows).sort_values('Dice', ascending=False)
    df.to_csv(f'{OUTPUT_ROOT}/table1_main.csv', index=False)
    display(df[['model', 'n_cores', 'n_folds', 'Dice', 'Dice_ci_low', 'Dice_ci_high',
                'Dice_core_sd', 'Dice_fold_sd', 'per_fold_dice']])
else:
    print("No complete baseline results yet.")

### Does the across-slide spread track label quality?

Correlates each fold's Dice with the mean reader count of its test slide.


In [ ]:
if 'proposed_frozense_sam' in cvs:
    ref_cv = cvs['proposed_frozense_sam']
    fold_rows = []
    for man, dice in zip(ref_cv.fold_manifests, ref_cv.fold_means('dice')):
        slide = man['groups']['test'][0]
        fold_rows.append({
            'fold': man['fold'], 'held_out_slide': slide,
            'test_cores': man['n_cores']['test'],
            'mean_readers': rbg['per_group'].get(slide, {}).get('mean_readers', float('nan')),
            'fold_dice': dice,
        })
    fdf = pd.DataFrame(fold_rows)
    fdf.to_csv(f'{OUTPUT_ROOT}/fold_vs_label_quality.csv', index=False)
    print(fdf.to_string(index=False))

    ok = fdf['mean_readers'].notna() & fdf['fold_dice'].notna()
    if ok.sum() >= 3 and fdf.loc[ok, 'mean_readers'].nunique() > 1:
        r = float(fdf.loc[ok, 'mean_readers'].corr(fdf.loc[ok, 'fold_dice']))
        print(f"\nPearson r between mean readers and fold Dice: {r:+.3f} (n={int(ok.sum())})")
        print("Weak. State that the confound was checked and not observed." if abs(r) < 0.4
              else "Material. Disclose it: part of the across-slide spread may be "
                   "annotation quality rather than generalisation.")
    else:
        print("\nReader count is constant across slides -- no confound to check.")
else:
    print("Needs the proposed model's six folds (STAGE 'baselines').")

### Paired significance tests

Two-sided Wilcoxon signed-rank tests on per-core Dice, Holm-corrected, with bootstrap confidence intervals.


In [ ]:
if 'proposed_frozense_sam' in cvs and len(cvs) > 1:
    comp = cv_comparison_table(list(cvs.values()), reference='proposed_frozense_sam')
    print(format_comparison(comp))
    cdf = pd.DataFrame(comp)
    cdf.to_csv(f'{OUTPUT_ROOT}/paired_tests.csv', index=False)
    display(cdf[['model_b', 'n_pairs', 'mean_a', 'mean_b', 'median_difference',
                 'p_value', 'p_holm', 'significant_at_0.05']])
else:
    print("Needs the proposed model and at least one baseline, complete.")

## 5b · The final model, chosen on validation

Two candidates (skip connections; skip connections plus fine-tuning of the last encoder stage) are compared on validation Dice only, and the better one becomes the final model, `improved_skip_partial` (FrozenSE-SAM). Both U-Nets are retrained under the same protocol: pretrained weights at 5×10⁻⁵, new weights at 5×10⁻⁴.


In [ ]:
from frozense.experiments import build_improvements

imp_cvs = {}
if STAGE in ('improve', 'report'):
    for exp in build_improvements(base):
        print(f"\n{'=' * 70}\n{exp.cfg.name}\n{exp.purpose}\n{'=' * 70}", flush=True)
        cv = run_cross_validation(
            exp.cfg, cores, device=DEVICE, verbose=True,
            train_missing=(STAGE == 'improve' and not RAN_IN_WORKERS), deadline=DEADLINE)
        if cv is not None:
            imp_cvs[exp.cfg.name] = cv
    print(f"\ncomplete: {sorted(imp_cvs)}")
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'improve' (trains) or 'report'")

In [ ]:
def mean_val_dice(name):
    vals = []
    for f in range(n_folds(cores)):
        p = f'{OUTPUT_ROOT}/{name}/fold{f}/fold_result.json'
        if not os.path.exists(p):
            return None
        vals.append(json.load(open(p))['history']['best_val_score'])
    return float(np.mean(vals))

if STAGE in ('improve', 'report'):
    candidates = ['proposed_frozense_sam', 'improved_skip', 'improved_skip_partial']
    val = {n: mean_val_dice(n) for n in candidates}
    print("mean VALIDATION Dice over 6 folds -- the selection criterion:")
    for n, v in val.items():
        print(f"  {n:<26}" + ('not complete' if v is None else f'{v:.4f}'))
    if all(v is not None for v in val.values()):
        WINNER = max(val, key=val.get)
        print(f"\nselected on validation: {WINNER}")
        if WINNER == 'proposed_frozense_sam':
            print("Neither change beat the original on validation, so table1_main.csv "
                  "stands as the result. Report both variants in the ablation table.")
        else:
            final = {WINNER: imp_cvs[WINNER]}
            for n in ('baseline_unet_pretrained_r2', 'baseline_unet_scratch_r2'):
                if n in imp_cvs:
                    final[n] = imp_cvs[n]
            native = next(e for e in build_baselines(base)
                          if e.cfg.name == 'baseline_mobilesam_native')
            cv = run_cross_validation(native.cfg, cores, device=DEVICE,
                                      verbose=False, train_missing=False)
            if cv is not None:
                final[native.cfg.name] = cv
            rows = cv_summary_table(list(final.values()), base)
            print()
            print(format_cv_table(sorted(rows, key=lambda r: -r['Dice'])))
            pd.DataFrame(rows).to_csv(f'{OUTPUT_ROOT}/table1_improved.csv', index=False)
            comp = cv_comparison_table(list(final.values()), reference=WINNER)
            print()
            print(format_comparison(comp))
            pd.DataFrame(comp).to_csv(f'{OUTPUT_ROOT}/paired_tests_improved.csv', index=False)
            if 'baseline_unet_scratch_r2' not in final:
                print("\n(baseline_unet_scratch_r2 not complete yet -- run 'improve' again.)")
    else:
        print("\nRun STAGE 'improve' again until all three have six folds.")

## 6 · Ablations

One-change ablations of the final model on folds 0, 2 and 4: SE blocks removed, an auxiliary signed-distance head, the focal Tversky loss and a soft Dice loss.


In [ ]:
from frozense.experiments import build_ablations, build_loss_ablations

abl_reference, ablation_cvs = None, {}
if STAGE in ('ablations', 'report'):
    train = (STAGE == 'ablations') and not RAN_IN_WORKERS
    abl_reference = run_cross_validation(
        FINAL, cores, folds=ABLATION_FOLDS, device=DEVICE, measure_eff=False,
        verbose=False, train_missing=train, deadline=DEADLINE)

    ref_fp = config_fingerprint(FINAL)
    exps = {e.cfg.name: e for e in build_ablations(FINAL) + build_loss_ablations(FINAL)}
    ordered = ([exps[n] for n in ABLATION_PRIORITY if n in exps]
               + [e for n, e in exps.items() if n not in ABLATION_PRIORITY])
    for exp in ordered:
        if config_fingerprint(exp.cfg) == ref_fp:
            print(f"{exp.cfg.name}: identical to the proposed model -- the reference covers it")
            continue
        print(f"\n{'=' * 70}\n{exp.cfg.name}\n{exp.purpose}\n{'=' * 70}", flush=True)
        cv = run_cross_validation(exp.cfg, cores, folds=ABLATION_FOLDS, device=DEVICE,
                                  measure_eff=False, verbose=True,
                                  train_missing=train, deadline=DEADLINE)
        if cv is not None:
            ablation_cvs[exp.cfg.name] = cv
    print(f"\ncomplete: {sorted(ablation_cvs)}")
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'ablations' (trains) or 'report'")

In [ ]:
if abl_reference is not None and ablation_cvs:
    abl_rows = cv_summary_table([abl_reference] + list(ablation_cvs.values()), FINAL)
    print(format_cv_table(abl_rows))
    pd.DataFrame(abl_rows).to_csv(f'{OUTPUT_ROOT}/table2_ablations.csv', index=False)

    abl_comp = cv_comparison_table([abl_reference] + list(ablation_cvs.values()),
                                   reference=FINAL.name)
    print()
    print(format_comparison(abl_comp))
    pd.DataFrame(abl_comp).to_csv(f'{OUTPUT_ROOT}/ablation_tests.csv', index=False)
else:
    print("No complete ablation results yet.")

## 7 · Empty-mask accounting

Counts, for every model, the patches on which the label, the prediction or both are empty. Surface distances are undefined when exactly one is empty; those patches are counted, not dropped.


In [ ]:
if cvs:
    acc = []
    for name, cv in cvs.items():
        c = cv.pooled.case_counts
        tot = sum(c.values())
        undef = c.get('gt_not_pred_empty', 0) + c.get('gt_empty_pred_not', 0)
        acc.append({
            'model': name, 'patches': tot,
            'both_present': c.get('both_present', 0),
            'both_empty': c.get('both_empty', 0),
            'missed_entirely': c.get('gt_not_pred_empty', 0),
            'false_alarm': c.get('gt_empty_pred_not', 0),
            'undefined_distance_pct': round(100 * undef / tot, 2) if tot else float('nan'),
        })
    adf = pd.DataFrame(acc)
    adf.to_csv(f'{OUTPUT_ROOT}/table3_empty_masks.csv', index=False)
    display(adf)
else:
    print("No complete baseline results yet.")

## 8 · Efficiency measurements

Parameters, inference latency, throughput and peak GPU memory for each model.


In [ ]:
if cvs:
    eff = []
    for name, cv in cvs.items():
        e = dict(cv.efficiency)
        e['model'] = name
        eff.append(e)
    edf = pd.DataFrame(eff)
    cols = [c for c in ['model', 'total', 'trainable', 'frozen', 'latency_ms_mean',
                        'throughput_patches_per_s', 'peak_memory_mb', 'device']
            if c in edf.columns]
    edf = edf[cols]
    edf.to_csv(f'{OUTPUT_ROOT}/table4_efficiency.csv', index=False)
    display(edf)
else:
    print("No complete baseline results yet.")

## 9 · Tables

Writes the result tables as CSV files into the output folder.


In [ ]:
# The methods paragraph, generated from the configuration that actually ran,
# so the text cannot drift from the code the way it did in the submission.
if 'proposed_frozense_sam' in cvs:
    ref_cv   = cvs['proposed_frozense_sam']
    m0       = ref_cv.fold_manifests[0]
    n_slides = sum(m0['n_groups'][k] for k in ('train', 'val', 'test'))
    n_fold   = len(ref_cv.fold_results)
    n_train  = m0['n_cores']['train']

    para = [
        'DATA AND PARTITIONING (draft text)',
        '',
        f'  {len(cores)} fully annotated TMA cores distributed over {n_slides} glass slides were',
        '  pooled from the released training and validation directories.',
        f'  Partitioning used leave-one-slide-out cross-validation: in each of {n_fold}',
        '  folds one slide was held out as the test partition, a second served as',
        '  the validation partition for checkpoint selection and early stopping,',
        f'  and {n_train} cores were sampled from the remaining slides for training. No',
        '  slide contributed cores to more than one partition in any fold, which',
        '  was asserted programmatically and recorded in a per-fold manifest',
        f'  listing every core in every partition. Over the {n_fold} folds each core was',
        '  evaluated exactly once, so core-level scores pool into a single',
        '  distribution for interval estimation and paired testing.',
        '',
        '  Because a tissue microarray slide carries cores from multiple patients,',
        '  slide-level grouping is coarser than patient-level grouping; the',
        '  released data provides no core-to-patient mapping, so a patient-level',
        '  disjointness claim cannot be made. Slide-level holdout does guarantee',
        '  that the test partition shares no fixation, staining or scanning batch',
        '  with training, which core-level partitioning cannot provide.',
    ]
    print(chr(10).join(para))
else:
    print("Needs the proposed model's six folds (STAGE 'baselines').")

In [ ]:
# Augmentation table: transforms, probabilities, magnitudes
aug_df = pd.DataFrame(base.aug.as_table_rows(),
                      columns=['Transform', 'Probability', 'Magnitude'])
aug_df.to_csv(f'{OUTPUT_ROOT}/table_augmentation.csv', index=False)
display(aug_df)

## 10 · Learning curve (validation)

Validation Dice with 5, 10, 20 and 35 training cores on folds 0, 2 and 4. This exploratory curve is superseded by the test-slide experiment in step 11.


In [ ]:
if STAGE in ('data_efficiency', 'report'):
    curve = []
    for m in (FINAL, UNET_R2):           # your model and the pretrained U-Net, same budgets
        rows = run_learning_curve(m, cores, sizes=CURVE_SIZES, folds=CURVE_FOLDS,
                                  device=DEVICE,
                                  train_missing=(STAGE == 'data_efficiency' and not RAN_IN_WORKERS),
                                  deadline=DEADLINE)
        curve += [dict(r, model=m.name) for r in (rows or [])]
    if curve:
        curve_df = pd.DataFrame(curve)
        curve_df.to_csv(f'{OUTPUT_ROOT}/data_efficiency.csv', index=False)
        summary = curve_df.pivot_table(index='n_train_cores', columns='model',
                                       values='val_dice', aggfunc=['mean', 'count'])
        print("mean VALIDATION Dice by training budget (3 folds each):")
        display(summary)
    else:
        print("No learning-curve results yet.")
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'data_efficiency' (trains) or 'report'")

## 11 · Five annotated cores, and pathology encoders

Test-slide results of the 5-core experiment (three draws, optimisation steps matched) and of FrozenSE-Phikon and FrozenSE-UNI, with paired tests against FrozenSE-SAM and the ImageNet U-Net.


In [ ]:
from scipy.stats import wilcoxon
extra_cvs = {}
if STAGE in ('extra', 'report'):
    for cfg, _, eff in EXTRA:
        print(f"\n{'=' * 70}\n{cfg.name}\n{'=' * 70}", flush=True)
        cv = run_cross_validation(cfg, cores, device=DEVICE, verbose=True, measure_eff=eff,
                                  train_missing=(STAGE == 'extra' and not RAN_IN_WORKERS),
                                  deadline=DEADLINE)
        if cv is not None:
            extra_cvs[cfg.name] = cv
    ref = {}
    for c in (FINAL, UNET_R2):
        cv = run_cross_validation(c, cores, device=DEVICE, verbose=False, train_missing=False)
        if cv is not None:
            ref[c.name] = cv

    # (1) five cores, test slides
    by_draw, per_core = [], {'frozense': {}, 'unet': {}}
    for d in LOWDATA_DRAWS:
        a, b = extra_cvs.get(f'lowdata5_frozense_d{d}'), extra_cvs.get(f'lowdata5_unet_d{d}')
        if a is None or b is None:
            print(f"draw {d}: not complete yet")
            continue
        for r in cv_comparison_table([a, b], reference=a.model_name):
            by_draw.append(dict(r, draw=d, frozense_dice=a.pooled.mean('dice'),
                                unet_dice=b.pooled.mean('dice')))
        for tag, cv in (('frozense', a), ('unet', b)):
            for c in cv.pooled.cores:
                per_core[tag].setdefault(c.core_id, []).append(c.dice)
    if by_draw:
        pd.DataFrame(by_draw).to_csv(f'{OUTPUT_ROOT}/lowdata_by_draw.csv', index=False)
        display(pd.DataFrame(by_draw)[['draw', 'frozense_dice', 'unet_dice'] +
                                      [k for k in ('p_value', 'p_holm') if k in by_draw[0]]])
        ids = sorted(set(per_core['frozense']) & set(per_core['unet']))
        fa = np.array([np.mean(per_core['frozense'][i]) for i in ids])
        ub = np.array([np.mean(per_core['unet'][i]) for i in ids])
        diff = fa - ub
        rng = np.random.default_rng(0)
        boot = rng.choice(diff, size=(2000, diff.size)).mean(1)
        row = {'n_draws': len(by_draw), 'n_cores': len(ids),
               'frozense_dice': fa.mean(), 'unet_dice': ub.mean(), 'mean_diff': diff.mean(),
               'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
               'median_diff': np.median(diff), 'p_wilcoxon': wilcoxon(fa, ub).pvalue,
               'cores_frozense_better': int((diff > 0).sum())}
        for tag, name in (('frozense', FINAL.name), ('unet', UNET_R2.name)):
            if name in ref:
                row[f'{tag}_35core_dice'] = ref[name].pooled.mean('dice')
                row[f'{tag}_retained'] = row[f'{tag}_dice'] / ref[name].pooled.mean('dice')
        pd.DataFrame([row]).to_csv(f'{OUTPUT_ROOT}/lowdata_pooled.csv', index=False)
        print("\nfive cores, averaged over draws, per core (TEST slides):")
        for k, v in row.items():
            print(f"  {k:<24}{v:.4f}" if isinstance(v, float) else f"  {k:<24}{v}")

    # (2) the same decoder on pathology encoders
    pf = [extra_cvs[n] for n in ('pfm_phikon', 'pfm_uni') if n in extra_cvs]
    if pf and FINAL.name in ref:
        allcv = [ref[FINAL.name]] + pf + ([ref[UNET_R2.name]] if UNET_R2.name in ref else [])
        rows = cv_summary_table(allcv, base)
        pd.DataFrame(rows).to_csv(f'{OUTPUT_ROOT}/pfm_main.csv', index=False)
        print(); print(format_cv_table(sorted(rows, key=lambda r: -r['Dice'])))
        comp = cv_comparison_table(allcv, reference=FINAL.name)
        pd.DataFrame(comp).to_csv(f'{OUTPUT_ROOT}/pfm_paired.csv', index=False)
        print(); print(format_comparison(comp))
        eff = [{'model': c.model_name, **(c.efficiency or {})} for c in allcv]
        pd.DataFrame(eff).to_csv(f'{OUTPUT_ROOT}/pfm_efficiency.csv', index=False)
    elif STAGE == 'extra':
        print("pathology-encoder runs not complete yet")
else:
    print(f"skipped in STAGE {STAGE!r} -- runs in 'extra' (trains) or 'report'")

## Progress, and packaging this session's results

Lists how many folds of each experiment are saved and packages the output folder as `frozense_runs.zip`. Attach that file as an input to the next session to continue.


In [ ]:
def folds_done(name, folds):
    return sum(os.path.exists(f'{OUTPUT_ROOT}/{name}/fold{f}/fold_result.json') for f in folds)

from frozense.experiments import build_ablations, build_loss_ablations, build_baselines
ALL_FOLDS = range(n_folds(cores))
print(f"{'experiment':<34}{'folds saved':>12}")
from frozense.experiments import build_improvements
for exp in build_baselines(base) + build_improvements(base):
    print(f"  {exp.cfg.name:<32}{folds_done(exp.cfg.name, ALL_FOLDS):>6} of {len(ALL_FOLDS)}")
for exp in build_ablations(FINAL) + build_loss_ablations(FINAL):
    if config_fingerprint(exp.cfg) == config_fingerprint(FINAL):
        continue
    print(f"  {exp.cfg.name:<32}{folds_done(exp.cfg.name, ABLATION_FOLDS):>6} of {len(ABLATION_FOLDS)}")
# Budgets equal to the main experiment's reuse its runs, so only the others are new.
curve_new = [(n, f) for n in CURVE_SIZES for f in CURVE_FOLDS if n != base.split.n_train_cores]
for m in (FINAL, UNET_R2):
    lc = sum(os.path.exists(f'{OUTPUT_ROOT}/learning_curve/{m.name}/{n}cores/fold{f}/history.json')
             for n, f in curve_new)
    print(f"  {'curve: ' + m.name:<32}{lc:>6} of {len(curve_new)}")

for c, _, _ in EXTRA:
    print(f"  {c.name:<32}{folds_done(c.name, ALL_FOLDS):>6} of {len(ALL_FOLDS)}")

if ON_KAGGLE:
    archive = shutil.make_archive('/kaggle/working/frozense_runs', 'zip', OUTPUT_ROOT)
    tables = '/kaggle/working/tables'
    os.makedirs(tables, exist_ok=True)
    for f in glob.glob(f'{OUTPUT_ROOT}/*.csv') + glob.glob(f'{OUTPUT_ROOT}/*.json'):
        shutil.copy(f, tables)
    # Kaggle's saved output is said to have a file-count limit of roughly 500;
    # the zip carries everything, so the loose folder goes once it gets large.
    n_files = sum(len(fs) for _, _, fs in os.walk(OUTPUT_ROOT))
    if n_files > 300:
        shutil.rmtree(OUTPUT_ROOT)
    print(f"\npackaged {n_files} files into {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")
    print("Next session: Add Input -> Your Work -> this notebook's latest output.")
if WORKER_PROBLEMS:
    print("\n" + "!" * 70)
    for problem in WORKER_PROBLEMS:
        print("  " + problem)
    print("Finished folds are saved; see the worker log above for the error.")
    print("!" * 70)
print(f"\nsession time: {(time.time() - SESSION_START) / 3600:.2f} h")